In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import matplotlib.colors as colors
# import holoviews as hv
import gzip
from matplotlib.patches import Rectangle

import matplotlib as mpl
from cycler import cycler
# hv.extension('bokeh')
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.signal import peak_widths
from scipy.signal import savgol_filter
from numpy.fft import rfft, rfftfreq

In [ ]:
from scipy.signal import savgol_filter
import gzip
import matplotlib as mpl
from cycler import cycler
# hv.extension('bokeh')
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.signal import peak_widths
from numpy.fft import rfft, rfftfreq

In [ ]:
def load1dFigS3(i):
    return np.loadtxt(f'../20221110 HMIA 13-2 QD/data/{i}/data.tsv')

def load2dFigS3(i, num):
    tmp = np.loadtxt(f'../20221110 HMIA 13-2 QD/data/{i}/data.tsv')
    numrows = np.floor(tmp.shape[0] / num)
    data_dict = {}
    for j in range(tmp.shape[1]):
        data_dict[str(j)] = tmp[:num*int(numrows), j].reshape((int(numrows), num)).transpose()
    return data_dict
# fast axis is column-wise 

In [ ]:
def fitG(deltaVg, G0, x):
    return G0*((1e-6+deltaVg*x)/(1e-6+np.sinh(deltaVg*x)))

In [ ]:
figS3 = plt.figure(figsize=(16, 4),constrained_layout=True)
gs = figS3.add_gridspec(1, 3, width_ratios=(1,1, 1))
# gs = GridSpec(3, 3, figure=fig1)
fS_ax1 = figS3.add_subplot(gs[:1, 0])
fS_ax2 = figS3.add_subplot(gs[:1, 1])
fS_ax3= figS3.add_subplot(gs[:1, 2])
# inset_ax = fig2.add_subplot(gs[:1, 2])
# f1_ax2cbar = fig1.add_subplot(gs[0, 3])
# plt.setp(f4_ax2.get_yticklabels(), visible=False)
# f3_ax2.set_title('gs[0, 3:]')
fS_ax1.text(-0.1, 1, "(a)", fontsize=14, va="bottom", ha="right", transform=fS_ax1.transAxes)
fS_ax2.text(-0.1, 1, "(b)", fontsize=14, va="bottom", ha="right", transform=fS_ax2.transAxes)
fS_ax3.text(-0.1, 1, "(c)", fontsize=14, va="bottom", ha="right", transform=fS_ax3.transAxes)


dat = load2dFigS3(157, 101)
vdc_slow = dat['1']
vplunger = dat['2']
curr = dat['3']

G = 25813*curr/5e-6
currpeak = np.zeros((8,vdc_slow.shape[1]))
currmax = np.zeros(vdc_slow.shape[1])
Gpeakfit = np.zeros(vdc_slow.shape[1])
widths = np.zeros((8,vdc_slow.shape[1]))
width_height = np.zeros((8,vdc_slow.shape[1]))
left = np.zeros((8,vdc_slow.shape[1]))
right = np.zeros((8,vdc_slow.shape[1]))
start=2

# ax2=ax[1].twinx()
ax1 = fS_ax2
ax = fS_ax1
# peaknum=1
# for j in range(len(start)):
for i in range(vdc_slow.shape[1]):
        
        peaks, _ = find_peaks(G[start:,i], prominence=0.005)
        currpeak[:len(peaks),i] = curr[start+peaks,i]
        currmax[i] = np.max(curr[start:,i])
#         print(vplunger[start + peaks[0],i])
#         phase = dat[:, 3]
#         v = 5e-6
#         vqpc = -1.988 - 0.002*i
        widths[:len(peaks), i], width_height[:len(peaks), i], left[:len(peaks), i], right[:len(peaks), i] = peak_widths(curr[start:,i], peaks, rel_height=0.5) 
#         ax.plot(1*0.03*i+curr[730:930]/v*25813, lw=0.75, label=str(vqpc)+' V', color='k')
        ax.plot(vplunger[start:,i], 1*0.01*i+G[start:,i], lw=0.75, label=str(vdc_slow[0,i])+' V', color='k')
#         ax.plot(1*0.03*i+curr[left[i]]/v*25813, lw=0.75, label=str(vqpc)+' V', color='k')
        # ax.plot(vplunger[start+peaks,i],1*0.01*i+G[start+peaks,i],'r*')
        # if i==vdc_slow.shape[1]-1:
        #       fS_ax3.plot(vplunger[start:,i],G[start:,i], 'b.-')

        for peaknum in [1]:
            Vg0 = vplunger[start+ peaks[peaknum], 0]
    #         print(Vg0)
            G0 = np.max(G[start+peaks[peaknum],i])
            alpha = 0.055
            Vg = vplunger[start + peaks[peaknum] - 20 : start + peaks[peaknum] + 20, 0 ]
            deltaVg = Vg - Vg0
            Gfit = G[start + peaks[peaknum] - 20 : start + peaks[peaknum] + 20, i]
            popt, pcov = curve_fit(fitG, deltaVg, Gfit, p0=[G0, 11000])
            Gpeakfit[i] = popt[0]
            print(alpha*1*(popt[1]**-1)*1e6/86)
            ax.plot(vplunger[start+peaks[peaknum],i],1*0.01*i+G[start+peaks[peaknum],i],'r*')
            # ax.plot(Vg, 1*0.01*i+fitG(deltaVg, popt[0], popt[1]), 'r--', lw=2)#, label='Fermi-Dirac')
#             print(np.sqrt(np.mean(curr[80:180]**2)))



# ax1.plot(vdc_slow[0,:], (widths[peaknum,:vdc_slow.shape[1]])*0.1, 'b.-')
ax1.plot(vdc_slow[0,:], Gpeakfit, 'b.-')
Gpeakfitsmooth = savgol_filter(Gpeakfit, 9, 2, deriv=1, delta = -5e-4)
ax2 = ax1.twinx()
ax2.plot(vdc_slow[0,:], Gpeakfitsmooth, 'r--')

ax1.set_ylabel('$G_{\\text{peak}}$ $(e^2/h)$')
# ax2.set_ylabel('Current noise in CB (fA)')
ax1.grid(ls='--')
ax.set_xlabel(r'$V_{pR}$ (V)')
# ax1.legend()
ax.set_ylabel('$G$ $(e^2/h)$, offset')
ax2.set_ylabel('$dG/dV_{\\text{qpc}}$ $(e^2/h/V)$')
ax1.set_xlabel('V $_{\\text{qpc}}$ (V)')
ax.grid(ls='--')

dat = load1dFigS3(159)
vplunger = dat[:,1]
curr = dat[:, 2]
G = 258138*curr/5e-6
fS_ax3.plot(vplunger, G, 'b.-')
dGdVgsmooth = savgol_filter(G, 9, 2, deriv=1, delta = -5e-5)
fS_ax33 = fS_ax3.twinx()
fS_ax33.plot(vplunger, dGdVgsmooth, 'r--')
fS_ax3.grid(ls='--', lw=0.4)
fS_ax3.set_xlim(-1.798, -1.7952)
fS_ax3.set_xlabel('$V_{{pR}}$ (V)')
fS_ax3.set_ylabel('$G$ $(e^2/h)$')
fS_ax33.set_ylabel('$dG/dV_{pR}$ $(e^2/h/V)$')

ax1.annotate("", xytext=(-3.6325, 0.015), xy=(-3.635, 0.015),
            arrowprops=dict(arrowstyle="->",color='b'))

ax2.annotate("", xytext=(-3.625, 1), xy=(-3.6225, 1),
            arrowprops=dict(arrowstyle="->",color='r', ls='--'))

fS_ax3.annotate("", xytext=(-1.7972, 0.05), xy=(-1.7977, 0.05),
            arrowprops=dict(arrowstyle="->",color='b'))

fS_ax33.annotate("", xytext=(-1.796, -100), xy=(-1.7955, -100),
            arrowprops=dict(arrowstyle="->",color='r', ls='--'))

# Define style dictionary
style_dict_red = {
    'color': 'red',
    'linewidth': 2,
    'fontsize': 14
}

# Define style dictionary
style_dict_blue = {
    'color': 'blue',
    'linewidth': 2,
    'fontsize': 14
}

# Apply style
spine = ax2.spines['right']
spine.set_color(style_dict_red['color'])
spine.set_linewidth(style_dict_red['linewidth'])

ax2.tick_params(axis='y', colors=style_dict_red['color'])
ax2.set_ylabel('$dG/dV_{\\text{qpc}}$ $(e^2/h/V)$', color=style_dict_red['color'], fontsize=style_dict_red['fontsize'])

# Apply style
spine = ax1.spines['left']
spine.set_color(style_dict_blue['color'])
spine.set_linewidth(style_dict_blue['linewidth'])

ax1.tick_params(axis='y', colors=style_dict_blue['color'])
ax1.set_ylabel('$G_{\\text{peak}}$ $(e^2/h)$', color=style_dict_blue['color'], fontsize=style_dict_blue['fontsize'])


# Apply style
spine = fS_ax33.spines['right']
spine.set_color(style_dict_red['color'])
spine.set_linewidth(style_dict_red['linewidth'])

fS_ax33.tick_params(axis='y', colors=style_dict_red['color'])
fS_ax33.set_ylabel('$dG/dV_{pR}$ $(e^2/h/V)$', color=style_dict_red['color'], fontsize=style_dict_red['fontsize'])

# Apply style
spine = fS_ax3.spines['left']
spine.set_color(style_dict_blue['color'])
spine.set_linewidth(style_dict_blue['linewidth'])

fS_ax3.tick_params(axis='y', colors=style_dict_blue['color'])
fS_ax3.set_ylabel('$G$ $(e^2/h)$', color=style_dict_blue['color'], fontsize=style_dict_blue['fontsize'])

# figS3.savefig('/Users/praveen/Library/CloudStorage/GoogleDrive-prvn@stanford.edu/Shared drives/GGG GDrive/QDots-2/Papers/Hybrid dot APL/FigS3.pdf', dpi=300)


